In [1]:
!pip install gradio openai pillow

In [2]:
import os

os.environ["NVIDIA_API_KEY"] = "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"

In [4]:
import os
import json
import base64
import mimetypes
from typing import Any, Dict

import gradio as gr
from openai import OpenAI
from PIL import Image

# -----------------------------------------------------------------------------
# CONFIG
# -----------------------------------------------------------------------------

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY", "")
BASE_URL = "https://integrate.api.nvidia.com/v1"

VISION_MODEL = "nvidia/nemotron-nano-12b-v2-vl"
TEXT_MODEL = "nvidia/nemotron-mini-4b-instruct"

client = OpenAI(
    base_url=BASE_URL,
    api_key=NVIDIA_API_KEY
)

# -----------------------------------------------------------------------------
# HELPERS
# -----------------------------------------------------------------------------

def prepare_image(path: str, max_side: int = 1024) -> str:
    """
    Resize large images before sending to the model.
    Returns path to resized image (or original if already small).
    """
    img = Image.open(path).convert("RGB")
    w, h = img.size

    if max(w, h) <= max_side:
        return path

    if w >= h:
        new_w = max_side
        new_h = int(h * max_side / w)
    else:
        new_h = max_side
        new_w = int(w * max_side / h)

    img = img.resize((new_w, new_h))
    resized_path = path + "_resized.jpg"
    img.save(resized_path, format="JPEG", quality=90)
    return resized_path


def file_to_data_url(path: str) -> str:
    mime = mimetypes.guess_type(path)[0] or "image/jpeg"
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime};base64,{b64}"


def extract_json(text: str, fallback: Dict[str, Any]) -> Dict[str, Any]:
    """
    More robust JSON extraction:
    - handles ```json ... ```
    - handles extra text before/after JSON
    """
    if not text:
        out = fallback.copy()
        out["_parse_status"] = "empty_response"
        return out

    cleaned = text.strip()

    # Remove markdown fences if present
    if "```" in cleaned:
        cleaned = cleaned.replace("```json", "```")
        cleaned = cleaned.replace("```JSON", "```")
        parts = cleaned.split("```")
        # Try fenced block contents first
        for part in parts:
            part = part.strip()
            if part.startswith("{") and part.endswith("}"):
                try:
                    parsed = json.loads(part)
                    out = fallback.copy()
                    out.update(parsed)
                    out["_parse_status"] = "parsed_from_fenced_block"
                    out["_raw_response"] = text
                    return out
                except Exception:
                    pass

    # Try direct full parse
    try:
        parsed = json.loads(cleaned)
        out = fallback.copy()
        out.update(parsed)
        out["_parse_status"] = "parsed_full_response"
        out["_raw_response"] = text
        return out
    except Exception:
        pass

    # Try first { ... last }
    try:
        start = cleaned.find("{")
        end = cleaned.rfind("}")
        if start != -1 and end != -1 and end > start:
            candidate = cleaned[start:end + 1]
            parsed = json.loads(candidate)
            out = fallback.copy()
            out.update(parsed)
            out["_parse_status"] = "parsed_substring"
            out["_raw_response"] = text
            return out
    except Exception:
        pass

    out = fallback.copy()
    out["_parse_status"] = "failed_to_parse_json"
    out["_raw_response"] = text
    return out


def call_text_agent(system_prompt: str, payload: Dict[str, Any], fallback: Dict[str, Any]) -> Dict[str, Any]:
    try:
        resp = client.chat.completions.create(
            model=TEXT_MODEL,
            temperature=0.1,
            max_tokens=220,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": json.dumps(payload)}
            ],
        )
        text = resp.choices[0].message.content or ""
        return extract_json(text, fallback)
    except Exception as e:
        out = fallback.copy()
        out["error"] = str(e)
        out["_parse_status"] = "text_agent_exception"
        return out

# -----------------------------------------------------------------------------
# AGENT 1: VISION AGENT (NEMOTRON VL)
# -----------------------------------------------------------------------------

def vision_agent(image_path: str, location: str, extra_context: str) -> Dict[str, Any]:
    system_prompt = """
You are the Vision Agent.

Describe only what is visually observable in the uploaded image.

Rules:
- Do not identify any person.
- Do not accuse anyone.
- Do not infer guilt or intent.
- If unclear, say so.
- Return ONLY valid JSON.
- Do not include markdown.
- Do not include extra explanation outside JSON.

Use exactly this schema:
{
  "scene_summary": "string",
  "objects_detected": ["string"],
  "visual_uncertainty": "low|medium|high"
}
"""

    fallback = {
        "scene_summary": "Unable to determine scene details reliably.",
        "objects_detected": [],
        "visual_uncertainty": "high"
    }

    try:
        if not image_path:
            out = fallback.copy()
            out["error"] = "No image path received."
            out["_parse_status"] = "missing_image_path"
            return out

        if not os.path.exists(image_path):
            out = fallback.copy()
            out["error"] = f"Image path does not exist: {image_path}"
            out["_parse_status"] = "image_not_found"
            return out

        prepared_path = prepare_image(image_path)
        mime = mimetypes.guess_type(prepared_path)[0] or "image/jpeg"
        size_bytes = os.path.getsize(prepared_path)
        data_url = file_to_data_url(prepared_path)

        resp = client.chat.completions.create(
            model=VISION_MODEL,
            temperature=0.1,
            max_tokens=250,
            messages=[
                {"role": "system", "content": system_prompt},
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": json.dumps({
                                "location": location,
                                "extra_context": extra_context
                            })
                        },
                        {
                            "type": "image_url",
                            "image_url": {"url": data_url}
                        }
                    ]
                }
            ]
        )

        text = resp.choices[0].message.content or ""
        parsed = extract_json(text, fallback)
        parsed["_debug"] = {
            "image_path": image_path,
            "prepared_path": prepared_path,
            "mime": mime,
            "size_bytes": size_bytes
        }
        return parsed

    except Exception as e:
        out = fallback.copy()
        out["error"] = str(e)
        out["_parse_status"] = "vision_agent_exception"
        out["_debug"] = {
            "image_path": image_path
        }
        return out

# -----------------------------------------------------------------------------
# AGENT 2: DETECTION AGENT (NEMOTRON)
# -----------------------------------------------------------------------------

def detection_agent(vision_result: Dict[str, Any], location: str, extra_context: str) -> Dict[str, Any]:
    system_prompt = """
You are the Detection Agent.

Based on the scene description and visible objects, determine whether there may be possible weapon presence.

Do not identify a person.
Do not infer intent.
Do not claim certainty if ambiguous.

Return ONLY valid JSON with exactly these keys:
{
  "possible_weapon_present": true,
  "weapon_type": "string",
  "confidence": 0.0
}
"""

    fallback = {
        "possible_weapon_present": False,
        "weapon_type": "unknown",
        "confidence": 0.2
    }

    payload = {
        "vision_result": vision_result,
        "location": location,
        "extra_context": extra_context
    }

    return call_text_agent(system_prompt, payload, fallback)

# -----------------------------------------------------------------------------
# AGENT 3: CONTEXT AGENT (NEMOTRON)
# -----------------------------------------------------------------------------

def context_agent(vision_result: Dict[str, Any], detection_result: Dict[str, Any], location: str, extra_context: str) -> Dict[str, Any]:
    system_prompt = """
You are the Context Agent.

Add environmental risk context for a public-safety screening event.

Use only the given inputs.
Do not invent detailed facts.

Return ONLY valid JSON with exactly these keys:
{
  "location_type": "string",
  "crowd_risk": "low|medium|high",
  "time_risk": "low|medium|high",
  "context_summary": "string"
}
"""

    fallback = {
        "location_type": "unknown",
        "crowd_risk": "medium",
        "time_risk": "medium",
        "context_summary": "Limited environmental context available."
    }

    payload = {
        "vision_result": vision_result,
        "detection_result": detection_result,
        "location": location,
        "extra_context": extra_context
    }

    return call_text_agent(system_prompt, payload, fallback)

# -----------------------------------------------------------------------------
# AGENT 4: TRIAGE AGENT (NEMOTRON)
# -----------------------------------------------------------------------------

def triage_agent(vision_result: Dict[str, Any], detection_result: Dict[str, Any], context_result: Dict[str, Any]) -> Dict[str, Any]:
    system_prompt = """
You are the Triage Agent.

Classify the possible public-safety risk level for human review.

Do not accuse any person.
Do not claim a crime occurred.
Use conservative wording.

Return ONLY valid JSON with exactly these keys:
{
  "threat_level": "low|medium|high",
  "urgency": "normal|elevated|immediate",
  "needs_human_review": true,
  "recommended_action": "string"
}
"""

    fallback = {
        "threat_level": "low",
        "urgency": "normal",
        "needs_human_review": True,
        "recommended_action": "monitor_and_review"
    }

    payload = {
        "vision_result": vision_result,
        "detection_result": detection_result,
        "context_result": context_result
    }

    return call_text_agent(system_prompt, payload, fallback)

# -----------------------------------------------------------------------------
# AGENT 5: DISPATCH AGENT (NEMOTRON)
# -----------------------------------------------------------------------------

def dispatch_agent(vision_result: Dict[str, Any], detection_result: Dict[str, Any], context_result: Dict[str, Any], triage_result: Dict[str, Any]) -> Dict[str, Any]:
    system_prompt = """
You are the Dispatch Agent.

Choose the most appropriate response route.

Allowed routes:
security_team
operations_center
campus_safety
emergency_review_queue

Return ONLY valid JSON with exactly these keys:
{
  "route_to": "security_team|operations_center|campus_safety|emergency_review_queue",
  "priority": "normal|urgent",
  "escalation_path": "string"
}
"""

    fallback = {
        "route_to": "operations_center",
        "priority": "normal",
        "escalation_path": "operator_review"
    }

    payload = {
        "vision_result": vision_result,
        "detection_result": detection_result,
        "context_result": context_result,
        "triage_result": triage_result
    }

    return call_text_agent(system_prompt, payload, fallback)

# -----------------------------------------------------------------------------
# AGENT 6: NOTIFICATION AGENT (NEMOTRON)
# -----------------------------------------------------------------------------

def notification_agent(vision_result: Dict[str, Any], detection_result: Dict[str, Any], context_result: Dict[str, Any], triage_result: Dict[str, Any], dispatch_result: Dict[str, Any]) -> Dict[str, Any]:
    system_prompt = """
You are the Notification Agent.

Write a short, human-readable public-safety alert.

Do not accuse anyone.
Use wording like "possible weapon presence" when appropriate.
Keep it concise.

Return ONLY valid JSON with exactly these keys:
{
  "message": "string"
}
"""

    fallback = {
        "message": "Possible safety incident detected. Human review recommended."
    }

    payload = {
        "vision_result": vision_result,
        "detection_result": detection_result,
        "context_result": context_result,
        "triage_result": triage_result,
        "dispatch_result": dispatch_result
    }

    return call_text_agent(system_prompt, payload, fallback)

# -----------------------------------------------------------------------------
# PIPELINE
# -----------------------------------------------------------------------------

def run_pipeline(image, location, extra_context):
    if image is None:
        err = {"error": "Please upload an image."}
        return (
            json.dumps(err, indent=2),
            json.dumps(err, indent=2),
            json.dumps(err, indent=2),
            json.dumps(err, indent=2),
            json.dumps(err, indent=2),
            json.dumps(err, indent=2),
            json.dumps(err, indent=2)
        )

    image_path = image

    a1 = vision_agent(image_path, location, extra_context)
    a2 = detection_agent(a1, location, extra_context)
    a3 = context_agent(a1, a2, location, extra_context)
    a4 = triage_agent(a1, a2, a3)
    a5 = dispatch_agent(a1, a2, a3, a4)
    a6 = notification_agent(a1, a2, a3, a4, a5)

    full = {
        "vision_agent": a1,
        "detection_agent": a2,
        "context_agent": a3,
        "triage_agent": a4,
        "dispatch_agent": a5,
        "notification_agent": a6
    }

    return (
        json.dumps(a1, indent=2),
        json.dumps(a2, indent=2),
        json.dumps(a3, indent=2),
        json.dumps(a4, indent=2),
        json.dumps(a5, indent=2),
        json.dumps(a6, indent=2),
        json.dumps(full, indent=2)
    )

# -----------------------------------------------------------------------------
# GRADIO UI
# -----------------------------------------------------------------------------

with gr.Blocks(title="Nemotron Multi-Agent Image Safety Demo") as demo:
    gr.Markdown("# Nemotron Multi-Agent Image Safety Demo")
    gr.Markdown("Upload an image, then the 6-agent Nemotron pipeline analyzes it.")

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(type="filepath", label="Upload Image")
            location_input = gr.Textbox(label="Location", value="North Gate")
            context_input = gr.Textbox(
                label="Optional Context",
                lines=3,
                value="Security camera frame from public entrance area."
            )
            run_btn = gr.Button("Run 6-Agent Pipeline")

        with gr.Column():
            out1 = gr.Textbox(label="1. Vision Agent Output", lines=14)
            out2 = gr.Textbox(label="2. Detection Agent Output", lines=10)
            out3 = gr.Textbox(label="3. Context Agent Output", lines=10)
            out4 = gr.Textbox(label="4. Triage Agent Output", lines=10)
            out5 = gr.Textbox(label="5. Dispatch Agent Output", lines=10)
            out6 = gr.Textbox(label="6. Notification Agent Output", lines=10)

    full_out = gr.Textbox(label="Full Pipeline JSON", lines=28)

    run_btn.click(
        fn=run_pipeline,
        inputs=[image_input, location_input, context_input],
        outputs=[out1, out2, out3, out4, out5, out6, full_out]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://467ba36efb4171269d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
